In [1]:
import os
import random
import pandas as pd
import cv2

from tqdm import tqdm

In [2]:

# ==========================================
# STEP 0: PARAMETERS & PATHS (sesuaikan)
# ==========================================
RAW_DIR = "../dataset"                          # berisi folder 'publik' & 'local'
PROCESSED_DIR = "../dataset_processed"        # output: train/ val/ test/ per kelas

SPLIT_RATIO = {
    "train": 0.8, 
    "val": 0.1, 
    "test": 0.1
}

os.makedirs(PROCESSED_DIR, exist_ok=True)

In [3]:
# ==========================================
# Struktur RAW_DIR:
# RAW_DIR/publik/<label>/*.jpg
# RAW_DIR/local/<label>/*.jpg
# ==========================================

all_images = {}
source_counts = {
    "public": {}, 
    "local": {}
}

for source in ["public", "local"]:
    source_path = os.path.join(RAW_DIR, source)
    if not os.path.isdir(source_path):
        continue

    for label in os.listdir(source_path):
        label_path = os.path.join(source_path, label)
        if not os.path.isdir(label_path):
            continue

        # Kumpulkan file yang valid
        valid_files = [
            os.path.join(label_path, fname)
            for fname in os.listdir(label_path)
            if fname.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        # Simpan ke all_images (gabungan keduanya)
        all_images.setdefault(label, []).extend(valid_files)

        # Catat jumlah per sumber
        source_counts[source][label] = len(valid_files)

# jika tidak ada data, hentikan
if not all_images:
    raise RuntimeError(f"Tidak menemukan data di {RAW_DIR}. Pastikan ada folder 'publik'/'local' dan subfolder label.")


In [4]:
# ==========================================
# Tabel ringkasan
# ==========================================
labels = sorted(list(all_images.keys()))
df_data = []

for lbl in labels:
    pub = source_counts["public"].get(lbl, 0)
    loc = source_counts["local"].get(lbl, 0)
    total = pub + loc
    df_data.append({
        "Kelas": lbl,
        "Public": pub,
        "Local": loc,
        "Total": total
    })

# Tampilkan ke terminal
pd.DataFrame(df_data)

,Kelas,Public,Local,Total
0,alur,187,68,255
1,lubang,952,63,1015
2,retak,806,232,1038
3,tidak_rusak,322,168,490


In [5]:
# ==========================================
# STEP 3: Shuffle, split, preprocess + rename hasil ke PROCESSED_DIR
# Struktur output:
# PROCESSED_DIR/train/<label>/...
# PROCESSED_DIR/val/<label>/...
# PROCESSED_DIR/test/<label>/...
# Format nama:
#   {source}_{label}_{nomor_urut:04d}.{ext}
# ==========================================

# Buat penghitung global untuk tiap sumber
counter = {"public": 1, "local": 1}

for label, files in all_images.items():
    random.shuffle(files)
    n_total = len(files)
    n_train = int(SPLIT_RATIO["train"] * n_total)
    n_val = int(SPLIT_RATIO["val"] * n_total)

    split_data = {
        "train": files[:n_train],
        "val": files[n_train:n_train + n_val],
        "test": files[n_train + n_val:]
    }

    for split, file_list in split_data.items():
        save_dir = os.path.join(PROCESSED_DIR, split, label)
        os.makedirs(save_dir, exist_ok=True)

        for fpath in tqdm(file_list, desc = f"Preprocess {split:<6} | {label:<12} : "):
            img = cv2.imread(fpath)
            if img is None:
                continue

            # Tentukan sumber: local / public
            fpath_lower = fpath.replace("\\", "/").lower()
            if "/local/" in fpath_lower:
                src = "local"
            elif "/public/" in fpath_lower:
                src = "public"
            else:
                src = "unknown"

            # Ambil ekstensi asli file
            _, ext = os.path.splitext(fpath)
            ext = ext.lower()  # biar rapi (contoh: .JPG -> .jpg)

            # Nama file baru (dengan ekstensi asli)
            new_name = f"{src}_{label}_{counter[src]:04d}{ext}"
            dst_path = os.path.join(save_dir, new_name)
            cv2.imwrite(dst_path, img)

            # Naikkan counter untuk sumber terkait
            counter[src] += 1

print("\nPreprocessing & rename selesai.")
print("Path:", PROCESSED_DIR)
print("Total:")
print(" - public:", counter["public"] - 1)
print(" - local :", counter["local"] - 1)


Preprocess train  | alur         : 100%|██████████| 204/204 [00:15<00:00, 12.88it/s]
Preprocess val    | alur         : 100%|██████████| 25/25 [00:01<00:00, 17.37it/s]
Preprocess test   | alur         : 100%|██████████| 26/26 [00:02<00:00, 11.06it/s]
Preprocess train  | lubang       : 100%|██████████| 812/812 [01:18<00:00, 10.40it/s]
Preprocess val    | lubang       : 100%|██████████| 101/101 [00:10<00:00,  9.46it/s]
Preprocess test   | lubang       : 100%|██████████| 102/102 [00:10<00:00,  9.66it/s]
Preprocess train  | retak        : 100%|██████████| 830/830 [00:47<00:00, 17.63it/s]
Preprocess val    | retak        : 100%|██████████| 103/103 [00:04<00:00, 21.68it/s]
Preprocess test   | retak        : 100%|██████████| 105/105 [00:06<00:00, 15.72it/s]
Preprocess train  | tidak_rusak  : 100%|██████████| 392/392 [00:13<00:00, 29.83it/s]
Preprocess val    | tidak_rusak  : 100%|██████████| 49/49 [00:02<00:00, 23.25it/s]
Preprocess test   | tidak_rusak  : 100%|██████████| 49/49 [00:01<00:00,


Preprocessing & rename selesai.
Path: ../dataset_processed
Total:
 - public: 2267
 - local : 531


In [6]:
# ==========================================
# Tampilkan jumlah file per kelas di setiap split
# ==========================================
def count_per_class(split_dir):
    split_path = os.path.join(PROCESSED_DIR, split_dir)
    if not os.path.exists(split_path):
        return {}
    return {
        cls: len([f for f in os.listdir(os.path.join(split_path, cls))
                  if os.path.isfile(os.path.join(split_path, cls, f))])
        for cls in os.listdir(split_path)
        if os.path.isdir(os.path.join(split_path, cls))
    }

train_counts = count_per_class("train")
val_counts = count_per_class("val")
test_counts = count_per_class("test")

# Gabungkan ke dalam satu DataFrame
all_classes = sorted(set(train_counts.keys()) | set(val_counts.keys()) | set(test_counts.keys()))

df_split_summary = pd.DataFrame({
    "Kelas" : all_classes,
    "Train" : [train_counts.get(cls, 0) for cls in all_classes],
    "Val"   : [val_counts.get(cls, 0) for cls in all_classes],
    "Test"  : [test_counts.get(cls, 0) for cls in all_classes],
})

# Hitung total per kelas
df_split_summary["Total"] = df_split_summary["Train"] + df_split_summary["Val"] + df_split_summary["Test"]

# Cetak ke terminal
print("Total gambar:")
df_split_summary

Total gambar:


,Kelas,Train,Val,Test,Total
0,alur,204,25,26,255
1,lubang,812,101,102,1015
2,retak,830,103,105,1038
3,tidak_rusak,392,49,49,490
